# Spectrum encoder — redshift probes

Frozen `LowResPT` embeddings at $1<z\le3$: zero-shot $k$-NN and a closed-form
ridge probe on one shared, group-aware split. Writes `spectrum_bench.png`.

In [ ]:
import sys
import numpy as np
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import r2_score

from plotstyle import (SPEC_DIR, SPEC_OUT, DJA_FITS, C_IMAGE, C_KNN, use_style, style_axes, fs, save)
sys.path.insert(0, str(SPEC_DIR))
from model.low_res_pt import LowResPT
from data.dataset import LowResDataset
from data.datamodule import LowResDataModule

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MIN_SN50, MIN_Z, MAX_Z = 0, 1, 3
FRAC_VALID_PIX = 0.9
POOL = "concat"        # both probes read the flattened token sequence
SPLIT_FRAC, SEED = 0.5, 42
KNN_K = 5
ALPHAS = np.logspace(-8, 6, 43)


def alpha_edge(a, grid=ALPHAS):
    """Flag a penalty that sits on an end of the search grid."""
    return ("FLOOR" if a <= grid[0] * 1.001 else
            "CEIL" if a >= grid[-1] * 0.999 else "ok")

# Lowest val_hid_loss checkpoint of the probe run.
PROBE_CKPT = min((SPEC_OUT / "low_res_pt_1_2_micron_noz_cut_tokenweight"
                  / "version_0/checkpoints").glob("*.ckpt"),
                 key=lambda p: float(p.stem.split("val_hid_loss=")[-1]))

use_style(1.3)
print(f"device={DEVICE}\nckpt={PROBE_CKPT.name}")

## Embeddings and probes

The split is grouped by `objid` so repeat spectra of one object cannot straddle it.

In [ ]:
def metrics(y, yp):
    dz = (yp - y) / (1.0 + y)
    return dict(r2=r2_score(y, yp),
                snmad=1.4826 * np.median(np.abs(dz - np.median(dz))),
                out=float(np.mean(np.abs(dz) > 0.15)))


def _pool(raw, tvm, pool):
    """Token sequence -> one vector per object. 'concat' keeps the fixed grid."""
    vm3 = tvm.float().unsqueeze(-1)
    tok = raw * vm3
    if pool == "concat":
        return tok.reshape(tok.shape[0], -1)
    if pool == "mean":
        return tok.sum(1) / vm3.sum(1).clamp(min=1)
    raise ValueError(pool)


@torch.no_grad()
def compute_embeddings(enc, dataset, pool=POOL, batch_size=256):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0,
                        collate_fn=LowResDataModule._pad_collate)
    Xs, ys = [], []
    for b in loader:
        out = enc.compute_embedding_from_raw_spectrum(
            b["flux"].to(DEVICE), b["wavelength"].to(DEVICE), b["valid_mask"].to(DEVICE))
        Xs.append(_pool(out["patch_token"], out["token_valid_mask"], pool).cpu().numpy())
        ys.append(b["redshift"].numpy())
    return np.concatenate(Xs), np.concatenate(ys)


model = LowResPT.load_from_checkpoint(PROBE_CKPT, map_location=DEVICE).eval().to(DEVICE)
ds = LowResDataset(str(DJA_FITS), min_sn50=MIN_SN50, min_redshift=MIN_Z,
                   max_redshift=MAX_Z, frac_valid_pix=FRAC_VALID_PIX)
X, y = compute_embeddings(model, ds)

tr, va = next(GroupShuffleSplit(n_splits=1, train_size=SPLIT_FRAC,
                                random_state=SEED).split(X, y, groups=ds.objid))
assert not set(ds.objid[tr]) & set(ds.objid[va]), "objid leaked across the split"
X_tr, X_va, y_tr, y_va = X[tr], X[va], y[tr], y[va]

sc = StandardScaler().fit(X_tr)
Z_tr, Z_va = sc.transform(X_tr), sc.transform(X_va)
pred_knn = KNeighborsRegressor(n_neighbors=KNN_K,
                               weights="distance").fit(Z_tr, y_tr).predict(Z_va)
ridge = RidgeCV(alphas=ALPHAS).fit(Z_tr, y_tr)
pred_ridge = ridge.predict(Z_va)

print(f"X={X.shape}  train={len(tr)}  test={len(va)}")
print(f"ridge alpha={ridge.alpha_:.2e} {alpha_edge(ridge.alpha_)}")
for tag, p in [("zero-shot kNN", pred_knn), ("linear probe", pred_ridge)]:
    m = metrics(y_va, p)
    print(f"{tag:16s} R2={m['r2']:.3f}  sigma_NMAD={m['snmad']:.4f}  "
          f"out={m['out']:.1%}")

## Figure

In [ ]:
def panel(ax, y, yp, title, color, ylabel=True, legend=False):
    m = metrics(y, yp)
    zl = np.linspace(MIN_Z, MAX_Z, 100)
    ax.scatter(y, yp, s=6, alpha=0.3, edgecolors="none", color=color)
    ax.fill_between(zl, zl - 0.15 * (1 + zl), zl + 0.15 * (1 + zl), alpha=0.12,
                    color="gray", label=r"$\pm0.15(1+z)$")
    ax.set_xlabel(r"$z_{\rm true}$")
    if ylabel:
        ax.set_ylabel(r"$z_{\rm pred}$")
    ax.text(0.97, 0.03, f"{title}\n"
            + rf"$R^2$={m['r2']:.3f}   $\sigma_{{\rm NMAD}}$={m['snmad']:.4f}",
            transform=ax.transAxes, ha="right", va="bottom", fontsize=fs(10), color=color,
            fontweight="bold",
            bbox=dict(facecolor="white", alpha=0.7, pad=3.0, edgecolor="none"))
    off = 0.05
    ax.set_xlim(MIN_Z - off, MAX_Z + off)
    ax.set_ylim(MIN_Z - 4 * off, MAX_Z + 4 * off)
    ax.set_box_aspect(1)
    if legend:
        ax.legend(fontsize=fs(12), loc="upper left", frameon=False)
    style_axes(ax)


fig, (ax_zs, ax_lp) = plt.subplots(1, 2, figsize=(11, 5.2))
panel(ax_zs, y_va, pred_knn, f"Zero-shot kNN (k={KNN_K})", C_KNN, legend=True)
panel(ax_lp, y_va, pred_ridge, "Linear probe", C_IMAGE)
for ax in (ax_zs, ax_lp):
    ax.set_xticks([1.0, 1.5, 2.0, 2.5, 3.0])

fig.patch.set_alpha(0.0)
fig.tight_layout()
save(fig, "spectrum_bench")
plt.show()